In [1]:
#!pip install tensorflow_datasets

In [2]:
from __future__ import absolute_import, division, print_function, unicode_literals
import functools

import numpy as np
import tensorflow as tf
#import tensorflow_datasets as tfds

In [3]:
TRAIN_DATA_URL = "https://storage.googleapis.com/tf-datasets/titanic/train.csv"
TEST_DATA_URL = "https://storage.googleapis.com/tf-datasets/titanic/eval.csv"

train_file_path = tf.keras.utils.get_file("train.csv", TRAIN_DATA_URL)
test_file_path = tf.keras.utils.get_file("eval.csv", TEST_DATA_URL)

# Make numpy values easier to read.
np.set_printoptions(precision=3, suppress=True)
#
!head {train_file_path}

survived,sex,age,n_siblings_spouses,parch,fare,class,deck,embark_town,alone
0,male,22.0,1,0,7.25,Third,unknown,Southampton,n
1,female,38.0,1,0,71.2833,First,C,Cherbourg,n
1,female,26.0,0,0,7.925,Third,unknown,Southampton,y
1,female,35.0,1,0,53.1,First,C,Southampton,n
0,male,28.0,0,0,8.4583,Third,unknown,Queenstown,y
0,male,2.0,3,1,21.075,Third,unknown,Southampton,n
1,female,27.0,0,2,11.1333,Third,unknown,Southampton,n
1,female,14.0,1,0,30.0708,Second,unknown,Cherbourg,n
1,female,4.0,1,1,16.7,Third,G,Southampton,n


In [4]:
CSV_COLUMNS = ['survived', 'sex', 'age', 'n_siblings_spouses', 'parch', 'fare', 'class', 'deck', 'embark_town', 'alone']

#dataset = tf.data.experimental.make_csv_dataset(train_file_path,column_names=CSV_COLUMNS,batch_size = 32)
#
LABEL_COLUMN = 'survived'
LABELS = [0, 1]

In [5]:
def get_dataset(file_path):
    dataset = tf.data.experimental.make_csv_dataset(
      file_path,
      batch_size=32, # Artificially small to make examples easier to show.
      label_name=LABEL_COLUMN,
      na_value="?",
      num_epochs=1,
      ignore_errors=True)
    return dataset

raw_train_data = get_dataset(train_file_path)
raw_test_data = get_dataset(test_file_path)

In [6]:
examples, labels = next(iter(raw_train_data)) # Just the first batch.
print("EXAMPLES: \n", examples, "\n")
print("LABELS: \n", labels)

CATEGORIES = {
    'sex': ['male', 'female'],
    'class' : ['First', 'Second', 'Third'],
    'deck' : ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J'],
    'embark_town' : ['Cherbourg', 'Southhampton', 'Queenstown'],
    'alone' : ['y', 'n']
}

categorical_columns = []
for feature, vocab in CATEGORIES.items():
    cat_col = tf.feature_column.categorical_column_with_vocabulary_list(
        key=feature, vocabulary_list=vocab)
    categorical_columns.append(tf.feature_column.indicator_column(cat_col))

#categorical_columns

EXAMPLES: 
 OrderedDict([('sex', <tf.Tensor: shape=(32,), dtype=string, numpy=
array([b'male', b'male', b'female', b'male', b'male', b'male', b'male',
       b'male', b'male', b'male', b'male', b'male', b'female', b'male',
       b'female', b'male', b'female', b'male', b'male', b'male', b'male',
       b'male', b'male', b'male', b'male', b'male', b'male', b'male',
       b'male', b'female', b'male', b'female'], dtype=object)>), ('age', <tf.Tensor: shape=(32,), dtype=float32, numpy=
array([28. , 50. , 33. , 28. , 62. , 28. , 38. , 36. , 47. , 28. , 19. ,
       28. , 42. , 60. , 36. , 18. , 31. , 35. , 33. , 32. , 50. , 34.5,
       28. ,  2. , 28. , 28. , 71. , 60. , 28. , 28. , 41. , 34. ],
      dtype=float32)>), ('n_siblings_spouses', <tf.Tensor: shape=(32,), dtype=int32, numpy=
array([0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0,
       0, 3, 0, 0, 0, 1, 1, 0, 2, 0], dtype=int32)>), ('parch', <tf.Tensor: shape=(32,), dtype=int32, numpy=
array([0, 0, 0, 0, 0, 0, 

In [7]:
def process_continuous_data(mean, data):
    # Normalize data
    data = tf.cast(data, tf.float32) * 1/(2*mean)
    return tf.reshape(data, [-1, 1])

MEANS = {
    'age' : 29.631308,
    'n_siblings_spouses' : 0.545455,
    'parch' : 0.379585,
    'fare' : 34.385399
}

numerical_columns = []

for feature in MEANS.keys():
    num_col = tf.feature_column.numeric_column(feature, normalizer_fn=functools.partial(process_continuous_data, MEANS[feature]))
    numerical_columns.append(num_col)

#numerical_columns

In [8]:
# cores = multiprocessing.cpu_count()
# print(cores)

In [9]:
# cores = multiprocessing.cpu_count()
# print(cores)

preprocessing_layer = tf.keras.layers.DenseFeatures(categorical_columns + numerical_columns)

model = tf.keras.Sequential([
  preprocessing_layer,
  tf.keras.layers.Dense(128, activation='relu'),
  tf.keras.layers.Dense(128, activation='relu'),
  tf.keras.layers.Dense(1, activation='sigmoid'),
])

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy'])

train_data = raw_train_data.shuffle(500)
test_data = raw_test_data

model.fit(train_data, epochs=200)

Epoch 1/200
Extension horovod.torch has not been built: /usr/local/lib/python3.8/site-packages/horovod/torch/mpi_lib/_mpi_lib.cpython-38-x86_64-linux-gnu.so not found
If this is not expected, reinstall Horovod with HOROVOD_WITH_PYTORCH=1 to debug the build error.
Warning! MPI libs are missing, but python applications are still avaiable.
[2022-04-29 19:09:43.290 tensorflow-2-6-cpu--ml-g4dn-xlarge-9f21f927d4a41ee7e7ffcbf71830:313 INFO utils.py:27] RULE_JOB_STOP_SIGNAL_FILENAME: None
[2022-04-29 19:09:43.460 tensorflow-2-6-cpu--ml-g4dn-xlarge-9f21f927d4a41ee7e7ffcbf71830:313 INFO profiler_config_parser.py:111] Unable to find config at /opt/ml/input/config/profilerconfig.json. Profiler is disabled.
Consider rewriting this model with the Functional API.


Consider rewriting this model with the Functional API.


Consider rewriting this model with the Functional API.


Consider rewriting this model with the Functional API.


20/20 [==============================] - 1s 3ms/step - loss: 0.5985 - accuracy: 0.6938
Epoch 2/200
20/20 [==============================] - 0s 2ms/step - loss: 0.4780 - accuracy: 0.7879
Epoch 3/200
20/20 [==============================] - 0s 3ms/step - loss: 0.4320 - accuracy: 0.8230
Epoch 4/200
20/20 [==============================] - 0s 2ms/step - loss: 0.4229 - accuracy: 0.8166
Epoch 5/200
20/20 [==============================] - 0s 2ms/step - loss: 0.4124 - accuracy: 0.8214
Epoch 6/200
20/20 [==============================] - 0s 2ms/step - loss: 0.4053 - accuracy: 0.8278
Epoch 7/200
20/20 [==============================] - 0s 2ms/step - loss: 0.3969 - accuracy: 0.8293
Epoch 8/200
20/20 [==============================] - 0s 3ms/step - loss: 0.3927 - accuracy: 0.8341
Epoch 9/200
20/20 [==============================] - 0s 3ms/step - loss: 0.3879 - accuracy: 0.8341
Epoch 10/200
20/20 [==============================] - 0s 2ms/step - loss: 0.3806 - accuracy: 0.8485
Epoch 11/200
20/20 [=

In [10]:
test_loss, test_accuracy = model.evaluate(test_data)

print('\n\nTest Loss {}, Test Accuracy {}'.format(test_loss, test_accuracy))

Consider rewriting this model with the Functional API.


Consider rewriting this model with the Functional API.


9/9 [==============================] - 0s 3ms/step - loss: 0.6147 - accuracy: 0.8030


Test Loss 0.6147415041923523, Test Accuracy 0.8030303120613098


In [11]:
predictions = model.predict(test_data)

# Show some results
for prediction, survived in zip(predictions[:10], list(test_data)[0][1][:10]):
    print("Predicted survival: {:.2%}".format(prediction[0]),
        " | Actual outcome: ",
        ("SURVIVED" if bool(survived) else "DIED"))

Consider rewriting this model with the Functional API.


Consider rewriting this model with the Functional API.


Predicted survival: 100.00%  | Actual outcome:  DIED
Predicted survival: 58.09%  | Actual outcome:  SURVIVED
Predicted survival: 87.72%  | Actual outcome:  DIED
Predicted survival: 13.44%  | Actual outcome:  SURVIVED
Predicted survival: 98.27%  | Actual outcome:  DIED
Predicted survival: 18.43%  | Actual outcome:  SURVIVED
Predicted survival: 47.85%  | Actual outcome:  DIED
Predicted survival: 8.41%  | Actual outcome:  SURVIVED
Predicted survival: 7.74%  | Actual outcome:  DIED
Predicted survival: 22.94%  | Actual outcome:  DIED
